In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time

ARCHIVE_URL = "https://kworb.net/spotify/world_daily.html"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

rows = []

# -------------------------
# STEP 1: FETCH ARCHIVE
# -------------------------
archive_html = requests.get(ARCHIVE_URL, headers=HEADERS).text
archive_soup = BeautifulSoup(archive_html, "html.parser")

links = archive_soup.find_all("a")

# Collect only 2025 daily links
daily_links = []
for a in links:
    href = a.get("href", "")
    if "world_daily.html?date=2025-" in href:
        daily_links.append("https://kworb.net/spotify/" + href)

print(f"🔗 Found {len(daily_links)} daily pages for 2025")

# -------------------------
# STEP 2: SCRAPE EACH DAY
# -------------------------
for url in daily_links:
    try:
        date_str = url.split("date=")[-1]
        r = requests.get(url, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")

        table = soup.find("table")
        if not table:
            print(f"⚠️ No table {date_str}")
            continue

        for tr in table.find_all("tr")[1:]:
            tds = tr.find_all("td")
            if len(tds) < 6:
                continue

            artist_title = tds[2].text.strip()
            daily_streams = int(tds[5].text.replace(",", ""))

            if " - " in artist_title:
                artist, track = artist_title.split(" - ", 1)
            else:
                artist = "Unknown"
                track = artist_title

            rows.append([
                date_str,
                track.strip(),
                artist.strip(),
                daily_streams
            ])

        print(f"✅ {date_str}")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error {url}: {e}")

# -------------------------
# STEP 3: MONTHLY AGGREGATE
# -------------------------
df = pd.DataFrame(rows, columns=["date", "track", "artist", "streams"])
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

monthly = (
    df.groupby(["month", "track", "artist"])["streams"]
    .sum()
    .reset_index()
)

top_monthly = (
    monthly
    .sort_values(["month", "streams"], ascending=[True, False])
    .groupby("month")
    .head(1)
)

top_monthly.to_csv(
    "spotify_global_monthly_top_2025.csv",
    index=False
)

print("\n🎉 DONE")
print("📁 spotify_global_monthly_top_2025.csv created")


🔗 Found 0 daily pages for 2025

🎉 DONE
📁 spotify_global_monthly_top_2025.csv created
